# FinanceRAG · 삼성전자 재무 분석
강의 GPU 이미지에서 이 파일과 같은 폴더를 작업 디렉터리로 사용하는 새 커널을 열고 순서대로 실행합니다.

In [ ]:
import os
import sys
from pathlib import Path

course = Path(os.environ.get("COURSE_ROOT", "/workspace/course"))
assert (course / "src/lab.py").is_file(), "COURSE_ROOT를 강의 저장소 경로로 지정하세요."
from app import create_app
from service import CourseRAG
sys.path.insert(0, str(course / "src"))
import lab
import torch

assert torch.cuda.is_available(), "GPU 커널을 선택하세요."
engine = CourseRAG(lab)
print("모델", lab.cfg()["model"], "문서", len(lab.docs(show=False)), "LoRA", engine.adapter)

In [ ]:
app = create_app(engine)
url = lab.run_api(app)

### API로 같은 질문 비교
기본 모델과 연결 가능한 LoRA에 같은 질문을 보냅니다. 기준값은 3분기 연결 매출액 79.10조 원입니다. 답변의 분기, 단위, 인용 번호를 반환된 원문과 대조하세요.

In [ ]:
import requests

for enabled in ([False, True] if engine.adapter else [False]):
    response = requests.post("http://localhost:8000/api/chat", json={
        "q": "삼성전자의 2024년 3분기 연결 매출액은 얼마인가요?", "use_lora": enabled
    }, timeout=240)
    response.raise_for_status()
    result = response.json()
    assert result["answer"] and result["sources"]
    assert result["adapter"] == (engine.adapter if enabled else None)
    print(result["adapter"], result["elapsed_ms"], "ms")
    print(result["answer"])
    for i, src in enumerate(result["sources"], 1):
        print(i, src["title"], src["text"], src["url"])

사용을 마치면 아래 셀의 주석을 해제하고 서버를 종료하세요.

In [ ]:
# lab.stop_api()